# Embedding
  
Das Notebook dient dazu per:
  
- **CLIP**          CLIP -> 
- **AnyLoc**        DINOv2 -> Feature Aggregation -> Descriptor -> Retrival (Github: https://github.com/AnyLoc/Revisit-Anything.git)
- **EigenPlaces**   Backbone -> VPR-Descriptor -> Retrival (Github: https://github.com/gmberton/EigenPlaces.git)
- **MixVPR**        noch keine Idee (Mixed ansatz)
  
die Bilder in Vectorinformationen zu embedden


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor


def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")


def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None

N_IMAGES = 10
PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
IMAGE_PATH = Path.home() / "Downloads" / "images"

DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"
metadata = pd.read_parquet(DATA_PATH_META)

embedding_metadata = metadata[metadata["split"].isin(["database", "query"])].copy()
embedding_metadata = embedding_metadata.reset_index(drop=True)

if N_IMAGES is not None:
    if N_IMAGES <= 0:
        raise ValueError("N_IMAGES muss None oder größer Null sein")

    if N_IMAGES > len(embedding_metadata):
        raise ValueError(f"N_IMAGES darf nicht größer als {len(embedding_metadata)} sein ist aber {N_IMAGES} ")

    embedding_metadata = embedding_metadata.iloc[:N_IMAGES].copy()


image_paths = [
    IMAGE_PATH / f"{image_id}.jpg" for image_id in embedding_metadata["image_id"]
]

EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings"
EMBEDDING_DIR.mkdir(parents = True, exist_ok = True)




# Parameter: versioniert, damit alle im Team denselben Datensatz erzeugen.
CFG_FILE = find_upwards("config.yaml")
assert CFG_FILE, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(CFG_FILE.read_text())


# Plotting high resolution
plt.rcParams["figure.dpi"] = 300

# A GPU makes the image encoder roughly 20x faster, but nothing here *needs* one.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32

METHOD = CFG["vpr"]["method"]

MODEL_ID = CFG["vpr"]["model"]

count = len(list(IMAGE_PATH.glob("*.jpg"))) + len(list(IMAGE_PATH.glob("*.png")))


METHOD_DIR = EMBEDDING_DIR / f"{METHOD}"

print(f"Bilder:             {len(metadata):,}")
print(f"Bilder downloaded:  {count}")
print(f"Bildordner:         {IMAGE_PATH}")
print(f"Embedding-Ordner:   {EMBEDDING_DIR}")
print(f"running on:         {DEVICE}")
print(f"batch size:         {BATCH_SIZE}")
print(f"Method:             {METHOD}")
print(f"MODEL_ID:           {MODEL_ID}")


# Modell laden


In [ ]:
MODEL_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"


model = CLIPModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION).to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)


EMBEDDING_DIM = model.config.projection_dim


print(f"parameters:         {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
print(f"embedding size:     {EMBEDDING_DIM}")



# Embedding Creation

In [ ]:

def embed_images(image_paths, batch_size=32):


    emb = []
    for start in tqdm(range(0,len(image_paths), batch_size), desc = "CLIP embeddings"):
        batch_paths = image_paths[start:start + batch_size]
        
        images = []

        for path in batch_paths:
            with Image.open(path) as image:
                images.append(
                    image.convert("RGB")
                )
    
        inputs = processor(
            images = images,
            return_tensors = "pt"
        ).to(DEVICE)

        with torch.inference_mode():

            batch_embeddings = model.get_image_features(**inputs).pooler_output
            
            batch_embeddings = (
                torch.nn.functional.normalize(
                    batch_embeddings,
                    dim = -1
                )
            )

        emb.append(
            batch_embeddings.cpu().numpy()
        )

    return np.concatenate(
        emb,
        axis = 0
    )



embeddings = embed_images(
    image_paths,
    batch_size = BATCH_SIZE
)


print("Shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)

#assert len(embeddings) == len(embedding_metadata)
assert embeddings.shape[1] == EMBEDDING_DIM
assert np.isfinite(embeddings).all()

norms = np.linalg.norm(
    embeddings,
    axis = 1
)

print(f"Normalized Minimum: {norms.min():.2f}")
print(f"Normalized Maximum: {norms.max():.2f}")
print(f"Normalized Mittelwert: {norms.mean():.2f}")


# Embedding Speichern


In [ ]:
embedding_path = METHOD_DIR / f"{METHOD}_embeddings.npy"

metadata_path = METHOD_DIR / f"{METHOD}_metadata.parquet"



np.save(
    embedding_path,
    embeddings
)

embedding_metadata.to_parquet(
    metadata_path,
    index = False
)

print(f"Embeddings gespeichert in:  {embedding_path}")
print(f"Metadaten gespeichert in:   {metadata_path}")